# Mamba upscale en Google Colab — flujo MAST-only (sin Drive)

**Objetivo:** reentrenar Mamba en la GPU de Colab (T4 16 GB o A100 40 GB) — 4-10x más VRAM que la RTX 3050 4 GB local — y comparar contra el baseline lockeado (AUC test = 0.806).

**Cero subida manual de datos.** Colab descarga todo automáticamente:
1. Catálogo TOI de NASA Exoplanet Archive
2. Curvas de luz desde **MAST** vía `lightkurve`
3. Preprocesamiento
4. Splits por TIC ID
5. Entrenamiento
6. Evaluación sobre test sealed
7. Descarga directa de los checkpoints a tu PC vía `files.download()`

**Tiempos esperados:**
| Paso | T4 | A100 |
|---|---|---|
| Setup + instalación | ~5 min | ~5 min |
| Descarga MAST + preprocesamiento | ~1.5-2 h | ~1.5-2 h |
| Sanity overfit | ~5 min | ~3 min |
| Training real (50 ep) | ~45 min | ~15 min |
| Evaluación + descarga | ~3 min | ~3 min |
| **Total** | **~3 h** | **~2.5 h** |

**Aviso de Colab Free:** desconecta tras ~30 min de inactividad y tiene límite de ~12 h de runtime. Si tu sesión muere durante MAST, al reconectar **lightkurve cachea** los FITS ya descargados — solo continúa los que faltan.

**Aviso de checkpoints:** sin Drive, los archivos viven en `/content/` y se pierden al desconectar. La última sección descarga checkpoints a tu PC con `files.download()` — **corré esa celda antes de cerrar el navegador**.

---
## 1. Verificar GPU

Runtime → Change runtime type → **GPU**. Free te da T4 (16 GB). Pro te da A100 o V100.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

---
## 2. Instalar dependencias

El orden importa: `causal-conv1d` antes que `mamba-ssm` porque el segundo lo importa al compilar el kernel CUDA.

In [ ]:
# Dependencias de Mamba — compilación del kernel CUDA toma ~3-5 min
!pip install -q causal-conv1d>=1.4.0
!pip install -q mamba-ssm>=2.2.0

In [ ]:
# Fallback con versiones fijas — usar SOLO si la celda anterior rompe
# !pip install -q torch==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
# !pip install -q causal-conv1d==1.4.0
# !pip install -q mamba-ssm==2.2.6.post3

In [ ]:
# Resto de dependencias
!pip install -q lightkurve astropy pandas numpy matplotlib scikit-learn pyyaml tensorboard tqdm

In [ ]:
# Verificar que mamba-ssm importa y corre en GPU
try:
    from mamba_ssm import Mamba
    m = Mamba(d_model=64).cuda()
    x = torch.randn(2, 100, 64).cuda()
    out = m(x)
    print(f"OK — mamba-ssm funciona. Output shape: {out.shape}")
    del m, x, out
    torch.cuda.empty_cache()
except Exception as e:
    print(f"ERROR — mamba-ssm no funciona: {e}")
    print("Probar la celda de fallback y reiniciar el runtime (Runtime -> Restart).")

---
## 3. Clonar repositorio

Repo público — no necesita credenciales.

In [ ]:
%cd /content
!rm -rf mamba-exoplanet
!git clone https://github.com/JoseZum/mamba-exoplanet.git
%cd /content/mamba-exoplanet
!git log -1 --oneline

In [ ]:
# Instalar el paquete en modo editable
!pip install -q -e .

In [ ]:
# Verificar imports del proyecto
import sys
sys.path.insert(0, '/content/mamba-exoplanet/src')

from exoplanet.models.mamba_baseline import MambaBaseline
from exoplanet.training.runner import run_training
from exoplanet.training.config import load_config
print("OK — imports del proyecto funcionan")

---
## 4. Generar el dataset desde MAST

**Cuatro sub-pasos**, todos automáticos. Tarda ~1.5-2 horas en total.

Si tu runtime muere en el medio, **`lightkurve` cachea los FITS** en `~/.lightkurve/cache/`. Al reconectar y rerun, solo descarga los que faltan.

### 4.1 Catálogo TOI (rápido, ~10 s)
Descarga el CSV oficial del NASA Exoplanet Archive.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/get_data.py

### 4.2 Curvas de luz desde MAST (LENTO, ~1-1.5 h)
Descarga `_lc.fits` de cada TIC. Lightkurve resuelve qué sectores existen y concatena si hay múltiples.

**Mantené la pestaña activa** o Colab puede desconectar por inactividad.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/download_lightcurves.py 2>&1 | tail -30

### 4.3 Preprocesamiento (~20-30 min)
Normaliza por mediana de cada curva, interpola NaN, fija longitud a L=18,000 puntos. Genera un `.pt` por estrella en `data/processed/global/`.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/preprocess_global.py 2>&1 | tail -20

### 4.4 Splits train / val / test (rápido, ~5 s)
Estratificados por TIC ID — toda observación de una misma estrella vive en el mismo split, no en split por sector. Genera `data/splits/{train,val,test}_tics.csv`.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/make_splits.py

### 4.5 Verificar el dataset

In [ ]:
import os
import pandas as pd

PROCESSED = '/content/mamba-exoplanet/data/processed/global'
SPLITS = '/content/mamba-exoplanet/data/splits'

n_pt = len(os.listdir(PROCESSED))
print(f".pt files preprocesados: {n_pt}")

for split in ['train', 'val', 'test']:
    path = f'{SPLITS}/{split}_tics.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        cp = int((df.label == 1).sum())
        fp = int((df.label == 0).sum())
        print(f"  {split:5s}: N={len(df):4d}  CP={cp:4d}  FP={fp:4d}  (ratio CP/total={cp/len(df):.3f})")
    else:
        print(f"  {split}: NO existe {path}")

---
## 5. Smoke test — Mamba sobre tensores random

Verifica que el forward+backward del modelo corre en la GPU asignada. Si falla, es problema de entorno, no de datos.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/smoke_train_mamba.py

---
## 6. Sanity overfit

Forza a Mamba a memorizar 64 muestras sin regularización. **Debe llegar a val_auc > 0.97** o hay algo roto. Toma ~5-10 min en T4.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/train.py --config configs/mamba_sanity_overfit.yaml

In [ ]:
# Verificar el resultado del sanity
import glob
import pandas as pd

sanity_runs = sorted(glob.glob('experiments/*sanity*'))
if sanity_runs:
    last = sanity_runs[-1]
    metrics = pd.read_csv(f'{last}/metrics.csv')
    print(f"Último sanity run: {last}")
    print(metrics.tail(10))
    final_val_auc = metrics['val_auc_roc'].iloc[-1]
    print(f"\nval_auc final: {final_val_auc:.4f}")
    if final_val_auc >= 0.97:
        print("OK — sanity pasado, procedo al training real")
    else:
        print("WARNING — sanity NO pasa el umbral 0.97. Investigar antes de seguir.")
else:
    print("No hay runs de sanity. Volver a correr la celda anterior.")

---
## 7. Generar config upscaled

**Decisiones según GPU asignada:**

| Param | Local RTX 3050 (4GB) | Colab T4 (16GB) | Colab A100 (40GB) |
|---|---|---|---|
| `d_model` | 64 | 128 | 256 |
| `n_layers` | 4 | 6 | 8 |
| `batch_size` | 16 | 32 | 64 |
| `fp16` | true | true | true |
| `gradient_checkpointing` | true | false | false |

Editá la variable `GPU_TIER` según lo que muestre `nvidia-smi` del paso 1.

In [ ]:
# === EDITAR AQUÍ según tu GPU ===
GPU_TIER = 'T4'  # opciones: 'T4', 'L4', 'V100', 'A100'

presets = {
    'T4':   {'d_model': 128, 'n_layers': 6, 'batch_size': 32, 'lr': 1.0e-3},
    'L4':   {'d_model': 128, 'n_layers': 6, 'batch_size': 32, 'lr': 1.0e-3},
    'V100': {'d_model': 128, 'n_layers': 6, 'batch_size': 32, 'lr': 1.0e-3},
    'A100': {'d_model': 256, 'n_layers': 8, 'batch_size': 64, 'lr': 8.0e-4},
}
p = presets[GPU_TIER]
print(f"Preset {GPU_TIER}: {p}")

In [ ]:
import yaml

config = {
    'experiment': {
        'name': f'mamba_colab_{GPU_TIER.lower()}',
        'seed': 42,
        'output_dir': 'experiments',
    },
    'data': {
        'train_csv': 'data/splits/train_tics.csv',
        'val_csv': 'data/splits/val_tics.csv',
        'processed_dir': 'data/processed/global',
        'batch_size': p['batch_size'],
        'num_workers': 2,
    },
    'model': {
        'type': 'mamba_baseline',
        'params': {
            'in_channels': 1,
            'd_model': p['d_model'],
            'd_state': 16,
            'd_conv': 4,
            'expand': 2,
            'n_layers': p['n_layers'],
            'dropout': 0.1,
            'standardize': True,
        },
    },
    'training': {
        'optimizer': {
            'type': 'adamw',
            'lr': p['lr'],
            'weight_decay': 1.0e-4,
        },
        'scheduler': {
            'type': 'cosine',
            't_max': 50,
            'eta_min': 1.0e-6,
        },
        'loss': {
            'type': 'bce',
            'pos_weight': 'balanced',
        },
        'epochs': 50,
        'fp16': True,
        'gradient_checkpointing': False,
        'grad_clip': 1.0,
    },
    'early_stopping': {
        'enabled': True,
        'metric': 'val_auc',
        'mode': 'max',
        'patience': 10,
    },
    'logging': {
        'tensorboard': True,
        'log_every_n_steps': 10,
    },
}

config_path = 'configs/mamba_colab.yaml'
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f"Config escrito a {config_path}")
!cat {config_path}

---
## 8. Sanity de VRAM antes de gastar GPU

Mide cuánta VRAM consume el modelo upscaled con un batch real. Si pasa de 80% del VRAM total, bajá `d_model` o `batch_size`.

In [ ]:
from exoplanet.models.mamba_baseline import MambaBaseline
import torch

model = MambaBaseline(
    in_channels=1,
    d_model=p['d_model'],
    d_state=16,
    d_conv=4,
    expand=2,
    n_layers=p['n_layers'],
    dropout=0.1,
    standardize=True,
).cuda()

n_params = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")
print(f"Tamaño FP32: {n_params * 4 / 1e6:.1f} MB")
print(f"Tamaño FP16: {n_params * 2 / 1e6:.1f} MB")

x = torch.randn(p['batch_size'], 1, 18000).cuda()
label = torch.randint(0, 2, (p['batch_size'],)).float().cuda()
batch = {
    'global_view': x, 'label': label,
    'local_view': None, 'scalar_features': None,
    'tic_id': torch.zeros(p['batch_size'], dtype=torch.long)
}

torch.cuda.reset_peak_memory_stats()
with torch.amp.autocast('cuda'):
    out = model(batch)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(out, label)
loss.backward()
peak_gb = torch.cuda.max_memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nForward+backward OK")
print(f"VRAM peak: {peak_gb:.2f} GB  /  {total_gb:.1f} GB  ({100*peak_gb/total_gb:.1f}%)")

del model, out, loss, x, label, batch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

---
## 9. Training real (~45 min T4 / ~15 min A100)

Corre el config upscaled hasta 50 épocas con early stopping en val_auc.

**Importante:** no cierres la pestaña.

In [ ]:
%cd /content/mamba-exoplanet
!python scripts/train.py --config configs/mamba_colab.yaml

In [ ]:
# Plot de la curva de aprendizaje
import glob
import pandas as pd
import matplotlib.pyplot as plt

colab_runs = sorted(glob.glob('experiments/*mamba_colab*'))
best_run = colab_runs[-1]
print(f"Último run: {best_run}")

metrics = pd.read_csv(f'{best_run}/metrics.csv')
print(metrics.tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(metrics['epoch'], metrics['train_loss'], label='train_loss')
axes[0].plot(metrics['epoch'], metrics['val_loss'], label='val_loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()
axes[0].set_title('Loss')

axes[1].plot(metrics['epoch'], metrics['val_auc_roc'], color='C2', label='val_auc (colab)')
axes[1].axhline(0.806, ls='--', color='red', alpha=0.5, label='Mamba ensemble local (0.806)')
axes[1].axhline(0.810, ls=':', color='orange', alpha=0.5, label='Mamba best seed local (0.810)')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('AUC'); axes[1].legend()
axes[1].set_title('Val AUC')
plt.tight_layout(); plt.show()

---
## 10. Multi-seed (opcional)

Si tenés tiempo, entrená 2-3 seeds adicionales para ensemble. Cada uno tarda lo mismo que el primero.

In [ ]:
# Descomentar para correr seeds adicionales
# !python scripts/train.py --config configs/mamba_colab.yaml --seed 123 --name-suffix seed123
# !python scripts/train.py --config configs/mamba_colab.yaml --seed 789 --name-suffix seed789

---
## 11. Evaluación sobre test sealed

**Una sola vez al final.** Reporta AUC, F1, recall, precision, Brier.

In [ ]:
import glob
colab_runs = sorted(glob.glob('experiments/*mamba_colab*'))
best_run = colab_runs[-1]
print(f"Evaluando: {best_run}")

!python scripts/evaluate.py --run {best_run} --split test

In [ ]:
import json

with open(f'{best_run}/eval_test/metrics.json') as f:
    metrics_test = json.load(f)

print('=' * 60)
print('RESULTADOS EN TEST SEALED — Mamba upscale Colab')
print('=' * 60)
for k, v in metrics_test.items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v:.4f}")
    else:
        print(f"  {k:20s}: {v}")

auc = metrics_test.get('auc_roc', float('nan'))
delta_ensemble = auc - 0.806
delta_best = auc - 0.810

print('\nComparación vs baselines locales:')
print(f'  Mamba ensemble local (RTX 3050): 0.806')
print(f'  Mamba best seed local:           0.810')
print(f'  Mamba Colab upscale (este run):  {auc:.4f}')
print(f'  Delta vs ensemble: {delta_ensemble:+.4f}  ({"MEJORA" if delta_ensemble > 0 else "NO mejora"})')
print(f'  Delta vs best:     {delta_best:+.4f}  ({"MEJORA" if delta_best > 0 else "NO mejora"})')

---
## 12. Descargar el experimento completo a tu PC

**HACÉ ESTO ANTES DE CERRAR LA SESIÓN.** Comprime el run en un `.tar.gz` y dispara la descarga al browser.

El archivo va a tu carpeta Downloads del PC. Después podés moverlo a `mamba-exoplanet/experiments/` en tu repo local.

In [ ]:
import os

run_name = os.path.basename(best_run)
tarball = f'/content/{run_name}.tar.gz'

# Empaqueta el run completo: config, env_info, git_info, train.log, metrics.csv, checkpoints/, eval_test/
!cd /content/mamba-exoplanet/experiments && tar -czf {tarball} {run_name}

size_mb = os.path.getsize(tarball) / 1e6
print(f"Tarball: {tarball}  ({size_mb:.1f} MB)")

In [ ]:
# Disparar la descarga al PC
from google.colab import files
files.download(tarball)

### Descarga solo el `best.pt` (más rápido, ~1-5 MB)

Si no querés todo el experimento sino únicamente el checkpoint ganador:

In [ ]:
from google.colab import files
best_ckpt = f'{best_run}/checkpoints/best.pt'
print(f"Descargando: {best_ckpt}")
files.download(best_ckpt)

---
## 13. Próximos pasos

**Si el AUC mejora** vs el baseline local (0.806):

1. Descomprimir el tarball en tu PC:
   ```bash
   cd C:\Users\jfzum\Downloads\Proyecto-IA\mamba-exoplanet\experiments
   tar -xzf C:\Users\jfzum\Downloads\<run_name>.tar.gz
   ```
2. Actualizar `experiments/_LOCKED_BASELINE.json` con el nuevo run.
3. Rerun XAI y error analysis sobre el checkpoint nuevo:
   ```bash
   python scripts/run_xai.py --run experiments/<new_run>
   python scripts/error_analysis.py --predictions experiments/<new_run>/eval_test/predictions.csv
   ```
4. Actualizar la tabla principal en `paper/results/all_results.md` con la nueva fila.

**Si NO mejora** — también es resultado válido y publicable. Documentar en `BITACORA.md` como ablation negativa:

> El upscale de Mamba en Colab (T4/A100, d_model 2-4×, n_layers 1.5-2×, batch 2-4×) no superó el baseline lockeado en RTX 3050. Esto sugiere que el cuello de botella del proyecto es el tamaño del dataset (N=1576 etiquetados), no la capacidad arquitectónica del modelo. La hipótesis se respalda con un experimento controlado sobre 2-4× los parámetros.

Eso refuerza la conclusión del paper: la victoria de Mamba sobre CNN no es por capacidad, es por arquitectura — escalar capacidad no rinde más en este régimen.